[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badaouihakimou/sql-notebooks/blob/main/01-introduction-sql.ipynb)

# Introduction à SQL

Apprendre à interroger une base de données, une requête à la fois.

Ce notebook est un cours d'introduction à SQL, conçu pour être suivi de bout en bout.
Vous n'avez besoin d'aucune connaissance préalable en bases de données ni en Python.

Tout tourne dans le notebook grâce à [DuckDB](https://duckdb.org) : aucun serveur à
installer, aucune configuration, aucun fichier à télécharger. Le jeu de données est
généré automatiquement au démarrage.

---

## Ce que vous saurez faire à la fin

- lire une table et sélectionner les colonnes utiles (`SELECT`, `FROM`) ;
- filtrer les lignes selon des conditions, y compris sur les valeurs manquantes (`WHERE`, `IS NULL`) ;
- combiner plusieurs conditions sans tomber dans le piège classique (`AND`, `OR`, `NOT`) ;
- rechercher sur du texte partiel (`LIKE`) ;
- trier et limiter les résultats (`ORDER BY`, `LIMIT`) ;
- calculer des indicateurs (`COUNT`, `SUM`, `AVG`, `MIN`, `MAX`) ;
- calculer ces indicateurs par groupe (`GROUP BY`) et filtrer sur le résultat (`HAVING`) ;
- supprimer les doublons (`DISTINCT`) ;
- comprendre dans quel ordre SQL exécute réellement une requête.

## Comment l'utiliser

1. Exécutez les cellules de la section 0. Mise en route dans l'ordre. Une seule fois par session.
2. Lisez chaque section, puis exécutez les exemples et surtout, modifiez-les pour voir ce qui change.
3. Faites les exercices avant de regarder la correction. Se tromper fait partie de l'apprentissage.
4. Si vous redémarrez le kernel, relancez la section 0.

> Conventions du cours
> Les mots-clés SQL sont écrits en MAJUSCULES et les noms de colonnes en minuscules.
> SQL n'y est pas sensible, mais cette convention rend les requêtes bien plus lisibles.

---

## Sommaire

| # | Section | Mots-clés |
|---|---------|-----------|
| 0 | [Mise en route](#0) | installation, chargement des données |
| 1 | [Le jeu de données](#1) | `DESCRIBE` |
| 2 | [Lire une table](#2) | `SELECT`, `FROM` |
| 3 | [Filtrer les lignes](#3) | `WHERE`, `IS NULL`, `IN`, `BETWEEN` |
| 4 | [Combiner des conditions](#4) | `AND`, `OR`, `NOT` |
| 5 | [Chercher dans du texte](#5) | `LIKE` |
| 6 | [Trier et limiter](#6) | `ORDER BY`, `LIMIT` |
| 7 | [Calculer des indicateurs](#7) | `COUNT`, `SUM`, `AVG`, `MIN`, `MAX` |
| 8 | [Calculer par groupe](#8) | `GROUP BY` |
| 9 | [Supprimer les doublons](#9) | `DISTINCT` |
| 10 | [Filtrer sur un agrégat](#10) | `HAVING` |
| 11 | [L'ordre d'exécution](#11) | récapitulatif |
| 12 | [Exercices de synthèse](#12) | tout |
| — | [Aide-mémoire](#cheatsheet) | |

<a id="0"></a>
# 0. Mise en route

Trois cellules à exécuter dans l'ordre : installation, connexion, chargement des données.

Ce notebook fonctionne aussi bien sur Google Colab que sur votre machine
(Jupyter, VS Code). Vous n'avez rien à modifier.

In [48]:
# 0.1 Installation des dépendances
# duckdb  : le moteur de base de données (tourne en mémoire, sans serveur)
# duckdb-engine : le connecteur qui permet à JupySQL de parler à DuckDB
# jupysql : fournit la commande magique %%sql pour écrire du SQL dans une cellule
%pip install --quiet duckdb duckdb-engine jupysql pandas

In [49]:
# 0.2 Connexion et activation de la magie %%sql
import duckdb
import pandas as pd

# Une base DuckDB en mémoire : rien n'est écrit sur le disque,
# tout disparaît à la fermeture du notebook. Parfait pour apprendre.
con = duckdb.connect(":memory:")

%load_ext sql
%sql con --alias duckdb

# Les résultats sont retournés sous forme de DataFrame pandas (plus lisible).
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False
pd.options.display.float_format = "{:,.2f}".format


def q(sql: str) -> pd.DataFrame:
    """Alternative en pur Python, si la magie %%sql pose problème : q("SELECT ...")"""
    return con.execute(sql).df()


print("Connexion prête. Vous pouvez écrire du SQL avec %%sql.")

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
Connexion prête. Vous pouvez écrire du SQL avec %%sql.


In [50]:
# 0.3 Chargement des données
# Ce notebook est autonome : si `customers.csv` est introuvable, il génère
# lui-même un jeu de données équivalent. Vous n'avez rien à télécharger.
from pathlib import Path

CSV_NAME = "customers.csv"

# On cherche le fichier aux endroits habituels, du plus probable au moins probable.
csv_path = next(
    (p for p in [Path(CSV_NAME), Path("data") / CSV_NAME,
                 Path("..") / "data" / CSV_NAME, Path("/content") / CSV_NAME]
     if p.exists()),
    None,
)

if csv_path is None:
    # ---------------------------------------------------------------------
    # Génération d'un jeu de données FICTIF (aucune donnée personnelle réelle).
    # Déterministe : tout le monde obtient les mêmes résultats que les corrections.
    # Vous pouvez ignorer le détail de cette section, ce n'est pas du SQL.
    # ---------------------------------------------------------------------
    import csv as _csv, random, string
    from datetime import date, datetime, timedelta

    rng = random.Random(42)
    pick = lambda pairs: rng.choices([p[0] for p in pairs], [p[1] for p in pairs])[0]

    COUNTRIES = [("FR", 30), ("DE", 22), ("GB", 16), ("ES", 9), ("IT", 8),
                 ("CH", 5), ("BE", 4), ("NL", 3), ("PT", 2), ("LU", 1)]
    STATUSES = [("ACCOUNT_STATUS_ACTIVE", 70), ("ACCOUNT_STATUS_ONBOARDING", 20),
                ("ACCOUNT_STATUS_RECOVERY", 10)]
    JOBS = [("employee", 62), ("business owner", 18), ("unemployed", 12), (None, 8)]
    # niveau premium -> (poids, actifs moyens, écart-type)
    TIERS = {None: (65, 6500, 5000), "bronze": (20, 38000, 15000),
             "silver": (10, 105000, 30000), "gold": (5, 165000, 60000)}
    DOMAINS = [("gmail.com", 42), ("outlook.com", 16), ("yahoo.fr", 10), ("proton.me", 8),
               ("orange.fr", 6), ("web.de", 6), ("icloud.com", 6), ("hotmail.co.uk", 6)]
    DEVICES = [("iPhone 14", 20), ("iPhone 15", 14), ("Samsung Galaxy S23", 15),
               ("Google Pixel 7", 10), ("MacBook Pro", 14), ("Windows Desktop", 13),
               ("iPad Air", 8), ("Xiaomi Redmi Note 12", 6)]
    REFS = [(("reddit", "social", "r_personalfinance"), 14),
            (("reddit", "social", "r_eupersonalfinance"), 8),
            (("google", "cpc", "brand_search"), 20),
            (("google", "organic", "seo_blog"), 15),
            (("facebook", "social", "lookalike_fr"), 10),
            (("linkedin", "social", "b2b_founders"), 8),
            (("newsletter", "email", "monthly_digest"), 12),
            (("partner-blog", "referral", "affiliate_2024"), 8),
            ((None, None, None), 5)]  # trafic direct : pas d'URL
    FIRST = "alex marie lucas emma hugo lea jonas sofia liam chloe noah mila tom clara " \
            "ines leon anna yanis julia mateo sarah elias nina theo laura adam".split()
    LAST = "martin bernard dubois schmidt muller smith jones garcia rossi silva weber " \
           "fischer moreau laurent lefevre brown wilson keller conti ferrari nowak".split()

    TODAY = date(2025, 1, 1)
    rand_date = lambda a, b: a + timedelta(days=rng.randint(0, (b - a).days))
    seen_emails, rows = set(), []

    for _ in range(5000):
        status = pick(STATUSES)
        tier = pick([(k, v[0]) for k, v in TIERS.items()])

        while True:  # email unique
            email = (f"{rng.choice(FIRST)}{rng.choice(['.', '_', ''])}{rng.choice(LAST)}"
                     f"{'' if rng.random() < 0.6 else rng.randint(1, 99)}@{pick(DOMAINS)}")
            if email not in seen_emails:
                seen_emails.add(email)
                break

        created = rand_date(TODAY - timedelta(days=1095), TODAY - timedelta(days=1))
        # Le KYC n'est jamais validé tant que le compte est en onboarding.
        verified = None if status == "ACCOUNT_STATUS_ONBOARDING" else (
            datetime.combine(created, datetime.min.time())
            + timedelta(days=rng.randint(0, 21), hours=rng.randint(0, 23)))
        # premium_expires_at est NULL si et seulement si premium_tier est NULL.
        expires = None if tier is None else rand_date(TODAY - timedelta(days=90),
                                                      TODAY + timedelta(days=400))

        _, mean, sd = TIERS[tier]
        assets = max(0.0, rng.gauss(mean, sd))
        deposits = assets * rng.uniform(1.0, 1.9) + rng.uniform(0, 500)
        withdrawals = max(0.0, deposits - assets) * rng.uniform(0.4, 1.0)
        if status == "ACCOUNT_STATUS_ONBOARDING":  # rien déposé tant que le compte n'est pas ouvert
            assets = deposits = withdrawals = 0.0

        src, medium, campaign = pick(REFS)
        url = None if src is None else (
            f"https://www.novalto.com/signup"
            f"?utm_source={src}&utm_medium={medium}&utm_campaign={campaign}")

        device = pick(DEVICES)
        rows.append({
            "customer_id": "".join(rng.choices(string.ascii_uppercase + string.digits, k=12)),
            "date_of_birth": rand_date(TODAY - timedelta(days=27375),
                                       TODAY - timedelta(days=6570)).isoformat(),
            "country_code": pick(COUNTRIES),
            "email": email,
            "referral_url": url,
            "employment_status": pick(JOBS),
            "premium_tier": tier,
            "premium_expires_at": expires.isoformat() if expires else None,
            "account_status": status,
            "verified_at": verified.strftime("%Y-%m-%d %H:%M:%S") if verified else None,
            "total_asset_eur": round(assets, 2),
            "total_deposit_eur": round(deposits, 2),
            "total_withdrawal_eur": round(withdrawals, 2),
            "first_device": device,
            "last_device": device if rng.random() < 0.65 else pick(DEVICES),
        })

    csv_path = Path(CSV_NAME)
    with csv_path.open("w", newline="", encoding="utf-8") as fh:
        writer = _csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    print(f"Jeu de données de démonstration généré : {csv_path}")

# Chargement dans DuckDB : read_csv_auto devine les types tout seul.
con.execute(f"""
    CREATE OR REPLACE TABLE customers AS
    SELECT * FROM read_csv_auto('{csv_path.as_posix()}')
""")

n_rows = con.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
n_cols = len(con.execute("SELECT * FROM customers LIMIT 0").description)
print(f"✓ Table `customers` prête : {n_rows:,} lignes, {n_cols} colonnes.")

✓ Table `customers` prête : 5,000 lignes, 15 colonnes.


> Vous avez votre propre `customers.csv` ?
> Le notebook n'en a pas besoin il génère ses données tout seul. Mais si vous voulez
> utiliser votre fichier, deux façons de faire :
>
> - En local : placez-le à côté du notebook, ou dans un sous-dossier `data/`.
> - Sur Colab : exécutez la cellule ci-dessous, choisissez votre fichier,
>   puis relancez la cellule 0.3.
>
> ```python
> from google.colab import files
> uploaded = files.upload()
> ```
>
> Attention : sur Colab, les fichiers uploadés disparaissent à la fin de la session.

<a id="1"></a>
# 1. Le jeu de données

Tout au long du cours, vous travaillez pour Novalto, une banque en ligne fictive
présente dans une dizaine de pays européens.

Les différentes équipes (marketing, conformité, finance, produit) viennent vous voir
avec des questions métier. Votre travail : les traduire en requêtes SQL.

Une seule table pour l'instant : `customers`, où chaque ligne représente un client.

> Les données sont entièrement synthétiques : aucune donnée personnelle réelle
> n'est utilisée. Le générateur intégré à la cellule 0.3 est déterministe, donc
> vos résultats correspondront exactement à ceux des corrections.

### Le dictionnaire des colonnes

| Colonne | Type | Peut être `NULL` ? | Description |
|---|---|---|---|
| `customer_id` | texte | non | Identifiant unique du client, 12 caractères alphanumériques. Deux clients ne partagent jamais le même. |
| `date_of_birth` | date | non | Date de naissance (`YYYY-MM-DD`). Sert au contrôle de majorité et aux analyses par tranche d'âge. |
| `country_code` | texte | non | Pays de résidence sur 2 lettres (norme ISO 3166) : `FR`, `DE`, `GB`, `ES`, `IT`, `CH`… |
| `email` | texte | non | Adresse email. Sert d'identifiant de connexion et de canal de communication. |
| `referral_url` | texte | **oui** | URL d'arrivée du client, avec ses paramètres UTM (source, canal, campagne). `NULL` pour le trafic direct. |
| `employment_status` | texte | **oui** | Situation professionnelle : `employee`, `business owner`, `unemployed`. `NULL` si non renseignée. |
| `premium_tier` | texte | **oui** | Niveau d'abonnement : `bronze`, `silver`, `gold`. **`NULL` = pas d'abonnement.** |
| `premium_expires_at` | date | **oui** | Date d'expiration de l'abonnement. Toujours `NULL` si `premium_tier` est `NULL`. |
| `account_status` | texte | non | État du compte : `ACCOUNT_STATUS_ONBOARDING` (ouverture en cours), `ACCOUNT_STATUS_ACTIVE` (actif), `ACCOUNT_STATUS_RECOVERY` (récupération d'accès). |
| `verified_at` | horodatage | **oui** | Date de validation de l'identité (KYC). `NULL` tant que le client est en onboarding. |
| `total_asset_eur` | décimal | non | Valeur totale des actifs détenus sur la plateforme, en euros. |
| `total_deposit_eur` | décimal | non | Montant total déposé depuis l'inscription. |
| `total_withdrawal_eur` | décimal | non | Montant total retiré. |
| `first_device` | texte | non | Appareil utilisé lors de la première connexion. |
| `last_device` | texte | non | Appareil de la dernière connexion. |

Retenez surtout ceci : plusieurs colonnes peuvent contenir `NULL`, c'est-à-dire
*« l'information n'existe pas »*. `NULL` n'est ni `0`, ni une chaîne vide : c'est
l'absence de valeur, et SQL la traite de façon particulière. On y revient en section 3.

### Un premier coup d'œil

Avant toute analyse, le réflexe est toujours le même : regarder les données.
Deux commandes suffisent.

In [51]:
%%sql
-- La structure : nom et type de chaque colonne
DESCRIBE customers

,column_name,column_type,null,key,default,extra


In [52]:
%%sql
-- Un échantillon : les 10 premières lignes, toutes colonnes confondues
SELECT *
FROM customers
LIMIT 10

,customer_id,date_of_birth,country_code,email,referral_url,employment_status,premium_tier,premium_expires_at,account_status,verified_at,total_asset_eur,total_deposit_eur,total_withdrawal_eur,first_device,last_device
0,PQK51FPKH1DN,1982-04-05,IT,liam.garcia@gmail.com,https://www.novalto.com/signup?utm_source=link...,employee,None,NaT,ACCOUNT_STATUS_ACTIVE,2022-07-17 13:00:00,"8,673.80","10,791.06","1,559.88",MacBook Pro,MacBook Pro
1,5NQ4FMYZYCWT,1972-01-06,FR,tom.keller@yahoo.fr,None,employee,gold,2025-09-28,ACCOUNT_STATUS_ACTIVE,2024-01-30 06:00:00,"170,279.57","181,269.64","9,493.67",Windows Desktop,MacBook Pro
2,0T0PVN9ER14F,1964-05-28,IT,laura.garcia41@gmail.com,https://www.novalto.com/signup?utm_source=redd...,employee,bronze,2025-12-25,ACCOUNT_STATUS_ACTIVE,2022-05-23 18:00:00,"35,923.18","52,506.10","13,029.38",iPhone 14,iPhone 14
3,85JSG65KXVF1,1998-06-11,NL,anna_keller2@proton.me,https://www.novalto.com/signup?utm_source=goog...,None,gold,2025-10-22,ACCOUNT_STATUS_ACTIVE,2022-09-14 17:00:00,"124,353.13","154,536.11","14,093.38",Samsung Galaxy S23,MacBook Pro
4,J5PHT0HL9XPS,1960-11-28,FR,mila_garcia@web.de,https://www.novalto.com/signup?utm_source=goog...,employee,None,NaT,ACCOUNT_STATUS_ACTIVE,2022-06-14 23:00:00,"4,613.60","8,154.72","1,689.24",Google Pixel 7,Google Pixel 7
5,5UIRODXMO5BX,1958-11-18,FR,alex.nowak@gmail.com,https://www.novalto.com/signup?utm_source=goog...,business owner,None,NaT,ACCOUNT_STATUS_ACTIVE,2022-03-17 02:00:00,"6,666.48","9,890.57","2,228.65",MacBook Pro,iPhone 14
6,OR56FAO72KZ0,1999-11-30,GB,hugo_smith@gmail.com,https://www.novalto.com/signup?utm_source=news...,employee,None,NaT,ACCOUNT_STATUS_ACTIVE,2022-06-19 17:00:00,"12,434.13","18,491.08","2,761.71",iPhone 15,iPhone 15
7,YIE6IVWPVS7H,1978-03-29,FR,theobernard8@gmail.com,https://www.novalto.com/signup?utm_source=redd...,employee,None,NaT,ACCOUNT_STATUS_ACTIVE,2024-09-20 16:00:00,"10,693.39","12,688.92",894.13,MacBook Pro,iPhone 15
8,ZW9XA3KX7EED,1999-09-08,FR,lucas.lefevre73@hotmail.co.uk,https://www.novalto.com/signup?utm_source=redd...,employee,bronze,2025-06-19,ACCOUNT_STATUS_ACTIVE,2022-06-18 06:00:00,"34,671.71","62,598.31","17,362.42",Windows Desktop,Windows Desktop
9,FPZUE8YFBNTP,1953-10-21,DE,liambrown@icloud.com,https://www.novalto.com/signup?utm_source=redd...,unemployed,bronze,2025-05-07,ACCOUNT_STATUS_ACTIVE,2022-04-18 20:00:00,"72,742.57","127,060.63","32,598.11",Xiaomi Redmi Note 12,iPad Air


<a id="2"></a>
# 2. Lire une table : `SELECT` et `FROM`

Une requête SQL se lit comme une phrase : « sélectionne ces colonnes, dans cette table ».

```sql
SELECT colonne_1,
       colonne_2
FROM ma_table
```

- `SELECT` répond à la question « quelles colonnes ? »
- `FROM` répond à la question « dans quelle table ? »

C'est le squelette minimal : toute requête de lecture commence par là.

### Exemple 1 choisir ses colonnes

L'équipe marketing veut préparer un envoi. Elle a besoin de l'email et du pays de chaque client.

In [53]:
%%sql
SELECT
    email,
    country_code
FROM customers
LIMIT 10

,email,country_code
0,liam.garcia@gmail.com,IT
1,tom.keller@yahoo.fr,FR
2,laura.garcia41@gmail.com,IT
3,anna_keller2@proton.me,NL
4,mila_garcia@web.de,FR
5,alex.nowak@gmail.com,FR
6,hugo_smith@gmail.com,GB
7,theobernard8@gmail.com,FR
8,lucas.lefevre73@hotmail.co.uk,FR
9,liambrown@icloud.com,DE


### Exemple 2 tout sélectionner avec `*`

L'étoile `*` est un raccourci pour « toutes les colonnes ».

In [54]:
%%sql
SELECT *
FROM customers
LIMIT 5

,customer_id,date_of_birth,country_code,email,referral_url,employment_status,premium_tier,premium_expires_at,account_status,verified_at,total_asset_eur,total_deposit_eur,total_withdrawal_eur,first_device,last_device
0,PQK51FPKH1DN,1982-04-05,IT,liam.garcia@gmail.com,https://www.novalto.com/signup?utm_source=link...,employee,None,NaT,ACCOUNT_STATUS_ACTIVE,2022-07-17 13:00:00,"8,673.80","10,791.06","1,559.88",MacBook Pro,MacBook Pro
1,5NQ4FMYZYCWT,1972-01-06,FR,tom.keller@yahoo.fr,None,employee,gold,2025-09-28,ACCOUNT_STATUS_ACTIVE,2024-01-30 06:00:00,"170,279.57","181,269.64","9,493.67",Windows Desktop,MacBook Pro
2,0T0PVN9ER14F,1964-05-28,IT,laura.garcia41@gmail.com,https://www.novalto.com/signup?utm_source=redd...,employee,bronze,2025-12-25,ACCOUNT_STATUS_ACTIVE,2022-05-23 18:00:00,"35,923.18","52,506.10","13,029.38",iPhone 14,iPhone 14
3,85JSG65KXVF1,1998-06-11,NL,anna_keller2@proton.me,https://www.novalto.com/signup?utm_source=goog...,None,gold,2025-10-22,ACCOUNT_STATUS_ACTIVE,2022-09-14 17:00:00,"124,353.13","154,536.11","14,093.38",Samsung Galaxy S23,MacBook Pro
4,J5PHT0HL9XPS,1960-11-28,FR,mila_garcia@web.de,https://www.novalto.com/signup?utm_source=goog...,employee,None,NaT,ACCOUNT_STATUS_ACTIVE,2022-06-14 23:00:00,"4,613.60","8,154.72","1,689.24",Google Pixel 7,Google Pixel 7


> Bonne pratique
> `SELECT *` est pratique pour explorer, mais évitez-le dans une requête finale :
> vous rapatriez des colonnes inutiles, la requête est plus lente, et surtout le résultat
> change silencieusement si quelqu'un ajoute une colonne à la table.
> Nommez explicitement ce dont vous avez besoin.

> Le piège de la virgule
> Chaque colonne est séparée par une virgule, sauf la dernière. Si vous oubliez une virgule
> entre deux colonnes, SQL ne renvoie pas toujours d'erreur : il interprète le deuxième nom
> comme un *alias* du premier. Vous perdez une colonne sans vous en apercevoir.
>
> ```sql
> SELECT customer_id, email      -- ✗ deux colonnes attendues, une seule renvoyée,
>        country_code            --   renommée « country_code »
> FROM customers
> ```

### Renommer une colonne avec `AS`

`AS` donne un nom d'affichage à une colonne. Indispensable dès qu'on calcule quelque chose.

In [55]:
%%sql
SELECT
    customer_id     AS identifiant,
    country_code    AS pays,
    total_asset_eur AS actifs_eur
FROM customers
LIMIT 5

,identifiant,pays,actifs_eur
0,PQK51FPKH1DN,IT,"8,673.80"
1,5NQ4FMYZYCWT,FR,"170,279.57"
2,0T0PVN9ER14F,IT,"35,923.18"
3,85JSG65KXVF1,NL,"124,353.13"
4,J5PHT0HL9XPS,FR,"4,613.60"


## Exercice 2

L'équipe Conformité veut vérifier la qualité des données d'inscription.

Elle demande, pour chaque client : son identifiant, sa date de naissance et son
pays de résidence.

*Colonnes attendues :* `customer_id`, `date_of_birth`, `country_code`

<details>
<summary> Indice</summary>

Trois colonnes séparées par des virgules, dans une seule instruction `SELECT`.
Pas besoin de filtre à ce stade.
</details>

In [55]:
%%sql
-- Votre code ici

<details>
<summary>Voir la correction</summary>

```sql
SELECT
    customer_id,
    date_of_birth,
    country_code
FROM customers
```
</details>

In [56]:
%%sql
-- Correction
SELECT
    customer_id,
    date_of_birth,
    country_code
FROM customers
LIMIT 10

,customer_id,date_of_birth,country_code
0,PQK51FPKH1DN,1982-04-05,IT
1,5NQ4FMYZYCWT,1972-01-06,FR
2,0T0PVN9ER14F,1964-05-28,IT
3,85JSG65KXVF1,1998-06-11,NL
4,J5PHT0HL9XPS,1960-11-28,FR
5,5UIRODXMO5BX,1958-11-18,FR
6,OR56FAO72KZ0,1999-11-30,GB
7,YIE6IVWPVS7H,1978-03-29,FR
8,ZW9XA3KX7EED,1999-09-08,FR
9,FPZUE8YFBNTP,1953-10-21,DE


<a id="3"></a>
# 3. Filtrer les lignes : `WHERE`

`SELECT` choisit les colonnes. `WHERE` choisit les lignes.

```sql
SELECT colonne
FROM ma_table
WHERE condition
```

SQL parcourt la table ligne par ligne, évalue la condition, et ne garde que les lignes
pour lesquelles elle est vraie.

### Les opérateurs de comparaison

| Opérateur | Signification | Exemple |
|---|---|---|
| `=` | égal à | `country_code = 'FR'` |
| `!=` ou `<>` | différent de | `country_code != 'FR'` |
| `>` `>=` | supérieur (ou égal) à | `total_asset_eur >= 10000` |
| `<` `<=` | inférieur (ou égal) à | `date_of_birth < '1990-01-01'` |
| `IN (…)` | fait partie de la liste | `country_code IN ('FR', 'DE')` |
| `BETWEEN … AND …` | dans un intervalle (bornes incluses) | `total_asset_eur BETWEEN 1000 AND 5000` |
| `IS NULL` | la valeur est absente | `verified_at IS NULL` |
| `IS NOT NULL` | la valeur est présente | `premium_tier IS NOT NULL` |

> Les valeurs texte et les dates s'écrivent entre guillemets simples : `'FR'`, `'2024-01-01'`.
> Les nombres s'écrivent sans guillemets : `10000`.

### Exemple 1 égalité

L'équipe Conformité veut la liste des clients dont le compte est encore en cours d'ouverture.

In [58]:
%%sql
SELECT
    customer_id,
    email,
    account_status
FROM customers
WHERE account_status = 'ACCOUNT_STATUS_ONBOARDING'
LIMIT 10

,customer_id,email,account_status
0,OG72OY0IFZNB,sofiaschmidt@outlook.com,ACCOUNT_STATUS_ONBOARDING
1,J2O7S3KKV9RF,liam_brown32@outlook.com,ACCOUNT_STATUS_ONBOARDING
2,NVQHW06XKXB9,mila.conti@gmail.com,ACCOUNT_STATUS_ONBOARDING
3,X5T5P5O3O0LL,hugo.muller@yahoo.fr,ACCOUNT_STATUS_ONBOARDING
4,6BKLDSFR4MUE,emmafischer72@gmail.com,ACCOUNT_STATUS_ONBOARDING
5,XC08DTNQFLL7,sarahmartin85@web.de,ACCOUNT_STATUS_ONBOARDING
6,G8QYNPDPOC50,lucas_bernard@outlook.com,ACCOUNT_STATUS_ONBOARDING
7,3OQYFOQMF5L9,marieconti@gmail.com,ACCOUNT_STATUS_ONBOARDING
8,KNQT3WVEDN9U,leon_keller@gmail.com,ACCOUNT_STATUS_ONBOARDING
9,FYO1750CEHLX,theo.conti@outlook.com,ACCOUNT_STATUS_ONBOARDING


### Exemple 2 comparaison numérique

L'équipe Produit cible les clients détenant plus de 10 000 € d'actifs.

In [59]:
%%sql
SELECT
    customer_id,
    email,
    total_asset_eur
FROM customers
WHERE total_asset_eur >= 10000
LIMIT 10

,customer_id,email,total_asset_eur
0,5NQ4FMYZYCWT,tom.keller@yahoo.fr,"170,279.57"
1,0T0PVN9ER14F,laura.garcia41@gmail.com,"35,923.18"
2,85JSG65KXVF1,anna_keller2@proton.me,"124,353.13"
3,OR56FAO72KZ0,hugo_smith@gmail.com,"12,434.13"
4,YIE6IVWPVS7H,theobernard8@gmail.com,"10,693.39"
5,ZW9XA3KX7EED,lucas.lefevre73@hotmail.co.uk,"34,671.71"
6,FPZUE8YFBNTP,liambrown@icloud.com,"72,742.57"
7,5JU9BVM2P9E6,sofia.martin52@gmail.com,"33,915.71"
8,Q14J1RIPZIM6,emma_muller86@outlook.com,"10,690.08"
9,JCAT9MX2X18H,juliamartin@outlook.com,"89,997.43"


### Exemple 3 les valeurs manquantes : `IS NULL`

Vous voulez les clients pas encore vérifiés, c'est-à-dire ceux dont `verified_at` est vide.

C'est ici que `NULL` devient piégeux. `NULL` n'est égal à rien, pas même à lui-même.
La condition `verified_at = NULL` ne renvoie jamais aucune ligne et sans erreur, ce qui
la rend d'autant plus difficile à repérer. Il faut écrire `IS NULL`.

In [60]:
%%sql
-- ✗ Ne renvoie aucune ligne : on ne compare jamais avec = NULL
SELECT COUNT(*) AS lignes_trouvees
FROM customers
WHERE verified_at = NULL

,lignes_trouvees
0,0


In [61]:
%%sql
-- ✓ La bonne façon de tester l'absence de valeur
SELECT
    customer_id,
    email,
    account_status,
    verified_at
FROM customers
WHERE verified_at IS NULL
LIMIT 10

,customer_id,email,account_status,verified_at
0,OG72OY0IFZNB,sofiaschmidt@outlook.com,ACCOUNT_STATUS_ONBOARDING,NaT
1,J2O7S3KKV9RF,liam_brown32@outlook.com,ACCOUNT_STATUS_ONBOARDING,NaT
2,NVQHW06XKXB9,mila.conti@gmail.com,ACCOUNT_STATUS_ONBOARDING,NaT
3,X5T5P5O3O0LL,hugo.muller@yahoo.fr,ACCOUNT_STATUS_ONBOARDING,NaT
4,6BKLDSFR4MUE,emmafischer72@gmail.com,ACCOUNT_STATUS_ONBOARDING,NaT
5,XC08DTNQFLL7,sarahmartin85@web.de,ACCOUNT_STATUS_ONBOARDING,NaT
6,G8QYNPDPOC50,lucas_bernard@outlook.com,ACCOUNT_STATUS_ONBOARDING,NaT
7,3OQYFOQMF5L9,marieconti@gmail.com,ACCOUNT_STATUS_ONBOARDING,NaT
8,KNQT3WVEDN9U,leon_keller@gmail.com,ACCOUNT_STATUS_ONBOARDING,NaT
9,FYO1750CEHLX,theo.conti@outlook.com,ACCOUNT_STATUS_ONBOARDING,NaT


### Exemple 4 `IN` et `BETWEEN`

`IN` évite d'empiler les `OR`. `BETWEEN` évite d'empiler les comparaisons.

In [62]:
%%sql
SELECT
    customer_id,
    country_code,
    total_asset_eur
FROM customers
WHERE country_code IN ('FR', 'DE', 'GB')
  AND total_asset_eur BETWEEN 50000 AND 100000
LIMIT 10

,customer_id,country_code,total_asset_eur
0,FPZUE8YFBNTP,DE,"72,742.57"
1,JCAT9MX2X18H,GB,"89,997.43"
2,AO20U9ISYGJJ,FR,"91,033.96"
3,ML90PXRLZU6W,FR,"51,076.79"
4,M2LHGCSOTDHH,DE,"58,881.75"
5,E5EI28DS22WY,FR,"63,549.91"
6,KB9XN4M41KHT,GB,"70,704.16"
7,R838AMMZOHXS,FR,"53,228.60"
8,TR23QIUA08EK,FR,"95,164.20"
9,W69HZGOPJDN8,FR,"63,748.49"


## Exercice 3.1

L'équipe Marketing prépare une campagne ciblée sur les clients français.

Elle demande l'identifiant et l'email de tous les clients dont le pays de résidence
est la France.

*Colonnes attendues :* `customer_id`, `email`

<details>
<summary>Indice</summary>

Le code pays de la France est `'FR'`. N'oubliez pas les guillemets simples.
</details>

In [ ]:
%%sql
-- Votre code ici

In [ ]:
%%sql
-- Correction
SELECT
    customer_id,
    email
FROM customers
WHERE country_code = 'FR'
LIMIT 10

## Exercice 3.2

L'équipe Finance veut identifier les clients à fort potentiel : ceux qui ont un abonnement premium, quel que soit son niveau.

*Colonnes attendues :* `customer_id`, `email`, `premium_tier`, `premium_expires_at`

<details>
<summary>Indice</summary>

Relisez le dictionnaire : un client sans abonnement a `premium_tier` à `NULL`.
Vous cherchez donc l'inverse. Et souvenez-vous : `!= NULL` ne fonctionne pas.
</details>

In [ ]:
%%sql
-- Votre code ici

In [63]:
%%sql
-- Correction
SELECT
    customer_id,
    email,
    premium_tier,
    premium_expires_at
FROM customers
WHERE premium_tier IS NOT NULL
LIMIT 10

,customer_id,email,premium_tier,premium_expires_at
0,5NQ4FMYZYCWT,tom.keller@yahoo.fr,gold,2025-09-28
1,0T0PVN9ER14F,laura.garcia41@gmail.com,bronze,2025-12-25
2,85JSG65KXVF1,anna_keller2@proton.me,gold,2025-10-22
3,ZW9XA3KX7EED,lucas.lefevre73@hotmail.co.uk,bronze,2025-06-19
4,FPZUE8YFBNTP,liambrown@icloud.com,bronze,2025-05-07
5,5JU9BVM2P9E6,sofia.martin52@gmail.com,bronze,2025-08-27
6,JCAT9MX2X18H,juliamartin@outlook.com,silver,2025-02-13
7,7HV3IZKY22UB,inesrossi22@yahoo.fr,silver,2024-11-27
8,0J43D5IQVNB4,leon.laurent82@icloud.com,bronze,2025-01-01
9,8E8LD4D6AF58,nina.conti@web.de,silver,2025-10-19


<a id="4"></a>
# 4. Combiner des conditions : `AND`, `OR`, `NOT`

- `AND` : toutes les conditions doivent être vraies.
- `OR` : au moins une condition doit être vraie.
- `NOT` : inverse une condition.

```sql
WHERE condition_1 AND condition_2   -- les deux
WHERE condition_1 OR  condition_2   -- l'une ou l'autre (ou les deux)
WHERE NOT condition_1               -- l'inverse
```

### Exemple 1 `AND`

Les clients premium et basés en France. Les deux conditions doivent être remplies.

In [64]:
%%sql
SELECT
    customer_id,
    email,
    premium_tier,
    country_code
FROM customers
WHERE premium_tier IS NOT NULL
  AND country_code = 'FR'
LIMIT 10

,customer_id,email,premium_tier,country_code
0,5NQ4FMYZYCWT,tom.keller@yahoo.fr,gold,FR
1,ZW9XA3KX7EED,lucas.lefevre73@hotmail.co.uk,bronze,FR
2,5JU9BVM2P9E6,sofia.martin52@gmail.com,bronze,FR
3,74X2EK3ZEZQA,liam_wilson@outlook.com,bronze,FR
4,N9RR7S20CV3T,leon_bernard25@gmail.com,bronze,FR
5,AAKYTT3IMJ70,tom_moreau@proton.me,gold,FR
6,AO20U9ISYGJJ,ines_rossi@gmail.com,silver,FR
7,4PA8HYEXF7JX,jonas_rossi@gmail.com,gold,FR
8,9WRSB4GF16YE,lea_muller35@proton.me,bronze,FR
9,UF8438TGBCVA,liambernard29@outlook.com,bronze,FR


### Exemple 2 `OR`

Le Support Client veut les comptes en onboarding *ou* en recovery : les deux
situations nécessitent une intervention.

In [65]:
%%sql
SELECT
    customer_id,
    email,
    account_status
FROM customers
WHERE account_status = 'ACCOUNT_STATUS_ONBOARDING'
   OR account_status = 'ACCOUNT_STATUS_RECOVERY'
LIMIT 10

,customer_id,email,account_status
0,OG72OY0IFZNB,sofiaschmidt@outlook.com,ACCOUNT_STATUS_ONBOARDING
1,TGXN0GUO4KH2,theojones@gmail.com,ACCOUNT_STATUS_RECOVERY
2,J2O7S3KKV9RF,liam_brown32@outlook.com,ACCOUNT_STATUS_ONBOARDING
3,8E8LD4D6AF58,nina.conti@web.de,ACCOUNT_STATUS_RECOVERY
4,4UDOJEN0JNWN,liam.rossi62@hotmail.co.uk,ACCOUNT_STATUS_RECOVERY
5,NVQHW06XKXB9,mila.conti@gmail.com,ACCOUNT_STATUS_ONBOARDING
6,X5T5P5O3O0LL,hugo.muller@yahoo.fr,ACCOUNT_STATUS_ONBOARDING
7,6BKLDSFR4MUE,emmafischer72@gmail.com,ACCOUNT_STATUS_ONBOARDING
8,XC08DTNQFLL7,sarahmartin85@web.de,ACCOUNT_STATUS_ONBOARDING
9,G8QYNPDPOC50,lucas_bernard@outlook.com,ACCOUNT_STATUS_ONBOARDING


---

## Le piège à connaître : `AND` passe avant `OR`

C'est l'erreur la plus courante chez les débutants, et la plus vicieuse : elle ne
produit aucune erreur, juste un résultat faux.

Comme en mathématiques `×` est prioritaire sur `+`, en SQL `AND` est évalué avant `OR`.

Comparez les deux requêtes ci-dessous. L'intention est la même *« les clients gold ou silver,
mais seulement en France, Allemagne et Royaume-Uni »* mais une seule est correcte.

In [66]:
%%sql
-- ✗ SANS parenthèses
-- SQL lit en réalité :  premium_tier = 'gold'
--                       OR (premium_tier = 'silver' AND country_code IN (...))
-- Résultat : TOUS les clients gold remontent, y compris hors de ces trois pays.
SELECT COUNT(*) AS lignes_renvoyees
FROM customers
WHERE premium_tier = 'gold'
   OR premium_tier = 'silver'
  AND country_code IN ('FR', 'DE', 'GB')

,lignes_renvoyees
0,608


In [67]:
%%sql
-- ✓ AVEC parenthèses : l'intention est explicite et le résultat correct.
SELECT COUNT(*) AS lignes_renvoyees
FROM customers
WHERE (premium_tier = 'gold' OR premium_tier = 'silver')
  AND country_code IN ('FR', 'DE', 'GB')

,lignes_renvoyees
0,527


In [68]:
%%sql
-- ✓✓ Encore mieux : IN exprime la même chose, sans parenthèses ni ambiguïté.
SELECT COUNT(*) AS lignes_renvoyees
FROM customers
WHERE premium_tier IN ('gold', 'silver')
  AND country_code IN ('FR', 'DE', 'GB')

,lignes_renvoyees
0,527


> Le réflexe à prendre
> Dès que vous mélangez `AND` et `OR` dans un même `WHERE`, mettez des parenthèses,
> même quand elles semblent superflues. Elles coûtent deux caractères et vous évitent
> un bug silencieux.

## Exercice 4.1

L'équipe Conformité prépare un audit sur le marché allemand.

Elle a besoin des clients basés en Allemagne dont le compte est pleinement actif.

*Colonnes attendues :* `customer_id`, `email`, `account_status`

<details>
<summary> Indice</summary>

Deux conditions qui doivent être vraies en même temps. Le statut d'un compte
pleinement actif est `'ACCOUNT_STATUS_ACTIVE'`.
</details>

In [ ]:
%%sql
-- Votre code ici

In [69]:
%%sql
-- Correction
SELECT
    customer_id,
    email,
    account_status
FROM customers
WHERE country_code = 'DE'
  AND account_status = 'ACCOUNT_STATUS_ACTIVE'
LIMIT 10

,customer_id,email,account_status
0,FPZUE8YFBNTP,liambrown@icloud.com,ACCOUNT_STATUS_ACTIVE
1,7HV3IZKY22UB,inesrossi22@yahoo.fr,ACCOUNT_STATUS_ACTIVE
2,0J43D5IQVNB4,leon.laurent82@icloud.com,ACCOUNT_STATUS_ACTIVE
3,7PZE9VIFTTD9,laura_wilson1@yahoo.fr,ACCOUNT_STATUS_ACTIVE
4,8ZTV05X4W7KY,sarah_silva15@gmail.com,ACCOUNT_STATUS_ACTIVE
5,LH9BH811KERP,liam_laurent@outlook.com,ACCOUNT_STATUS_ACTIVE
6,0B6UGVS8Q349,anna.silva@gmail.com,ACCOUNT_STATUS_ACTIVE
7,7ZGF9Q5RDOQI,ines.rossi@gmail.com,ACCOUNT_STATUS_ACTIVE
8,OZD1AT7O75RH,leon.jones35@icloud.com,ACCOUNT_STATUS_ACTIVE
9,42YFKJFPNQVU,adam_fischer@outlook.com,ACCOUNT_STATUS_ACTIVE


## Exercice 4.2

L'équipe Marketing cible les clients les plus prestigieux. Sa règle métier :

> tous les clients gold et silver, mais uniquement sur les trois marchés
> principaux France, Allemagne, Royaume-Uni.

*Colonnes attendues :* `customer_id`, `email`, `premium_tier`, `country_code`

<details>
<summary> Indice</summary>

C'est exactement le piège de la section précédente. Deux façons correctes de l'écrire :
avec des parenthèses, ou avec deux `IN`.
</details>

In [ ]:
%%sql
-- Votre code ici

In [70]:
%%sql
-- Correction
SELECT
    customer_id,
    email,
    premium_tier,
    country_code
FROM customers
WHERE premium_tier IN ('gold', 'silver')
  AND country_code IN ('FR', 'DE', 'GB')
LIMIT 10

,customer_id,email,premium_tier,country_code
0,5NQ4FMYZYCWT,tom.keller@yahoo.fr,gold,FR
1,JCAT9MX2X18H,juliamartin@outlook.com,silver,GB
2,7HV3IZKY22UB,inesrossi22@yahoo.fr,silver,DE
3,8E8LD4D6AF58,nina.conti@web.de,silver,DE
4,31YV2STN7E36,milagarcia@gmail.com,silver,GB
5,AAKYTT3IMJ70,tom_moreau@proton.me,gold,FR
6,AO20U9ISYGJJ,ines_rossi@gmail.com,silver,FR
7,4PA8HYEXF7JX,jonas_rossi@gmail.com,gold,FR
8,WYOO5YBRJNM2,emma.conti37@web.de,gold,FR
9,UG0IT26YS445,adammoreau88@gmail.com,silver,FR


<a id="5"></a>
# 5. Chercher dans du texte : `LIKE`

`=` exige une correspondance exacte. Souvent, ce n'est pas ce qu'on veut :
on cherche un fragment à l'intérieur d'une chaîne.

`LIKE` compare avec un motif, construit à partir de deux jokers :

| Joker | Signifie | Exemple | Correspond à |
|---|---|---|---|
| `%` | n'importe quelle suite de caractères (même vide) | `'%reddit%'` | contient « reddit » |
| `_` | exactement un caractère | `'F_'` | `FR`, `FI`, `FO`… |

| Motif | Se lit |
|---|---|
| `'reddit%'` | commence par reddit |
| `'%gmail.com'` | finit par gmail.com |
| `'%reddit%'` | contient reddit |

### Exemple pourquoi `=` ne suffit pas

L'équipe Acquisition veut les clients venus de Reddit. Mais `referral_url` contient une
URL complète avec ses paramètres UTM, pas juste le mot « reddit ».

In [71]:
%%sql
-- Voyons d'abord à quoi ressemble la colonne
SELECT referral_url
FROM customers
WHERE referral_url IS NOT NULL
LIMIT 5

,referral_url
0,https://www.novalto.com/signup?utm_source=link...
1,https://www.novalto.com/signup?utm_source=redd...
2,https://www.novalto.com/signup?utm_source=goog...
3,https://www.novalto.com/signup?utm_source=goog...
4,https://www.novalto.com/signup?utm_source=goog...


In [72]:
%%sql
-- ✗ Aucune ligne : aucune URL n'est exactement égale à 'reddit'
SELECT COUNT(*) AS lignes_trouvees
FROM customers
WHERE referral_url = 'reddit'

,lignes_trouvees
0,0


In [73]:
%%sql
-- ✓ On cherche les URL qui *contiennent* reddit
SELECT
    customer_id,
    email,
    referral_url
FROM customers
WHERE referral_url LIKE '%reddit%'
LIMIT 10

,customer_id,email,referral_url
0,0T0PVN9ER14F,laura.garcia41@gmail.com,https://www.novalto.com/signup?utm_source=redd...
1,YIE6IVWPVS7H,theobernard8@gmail.com,https://www.novalto.com/signup?utm_source=redd...
2,ZW9XA3KX7EED,lucas.lefevre73@hotmail.co.uk,https://www.novalto.com/signup?utm_source=redd...
3,FPZUE8YFBNTP,liambrown@icloud.com,https://www.novalto.com/signup?utm_source=redd...
4,R4COWGZRIXA1,adamsmith37@outlook.com,https://www.novalto.com/signup?utm_source=redd...
5,Q14J1RIPZIM6,emma_muller86@outlook.com,https://www.novalto.com/signup?utm_source=redd...
6,7PZE9VIFTTD9,laura_wilson1@yahoo.fr,https://www.novalto.com/signup?utm_source=redd...
7,NVQHW06XKXB9,mila.conti@gmail.com,https://www.novalto.com/signup?utm_source=redd...
8,XC08DTNQFLL7,sarahmartin85@web.de,https://www.novalto.com/signup?utm_source=redd...
9,3G2QZW3C1QKB,mariemoreau33@proton.me,https://www.novalto.com/signup?utm_source=redd...


> Sensibilité à la casse
> `LIKE` distingue majuscules et minuscules dans la plupart des bases (`'%Reddit%'` ne
> trouverait rien ici). Pour ignorer la casse, mettez les deux côtés dans la même casse :
> `WHERE LOWER(referral_url) LIKE '%reddit%'`.
> DuckDB propose aussi `ILIKE`, insensible à la casse pratique, mais non standard.

## Exercice 5

L'équipe CRM segmente les clients par fournisseur d'email et commence par Gmail.

Elle demande les clients dont l'adresse email se termine par `gmail.com`.

*Colonnes attendues :* `customer_id`, `email`

<details>
<summary> Indice</summary>

« Se termine par » : le joker se place avant le texte recherché.
</details>

In [ ]:
%%sql
-- Votre code ici

In [74]:
%%sql
-- Correction
SELECT
    customer_id,
    email
FROM customers
WHERE email LIKE '%gmail.com'
LIMIT 10

,customer_id,email
0,PQK51FPKH1DN,liam.garcia@gmail.com
1,0T0PVN9ER14F,laura.garcia41@gmail.com
2,5UIRODXMO5BX,alex.nowak@gmail.com
3,OR56FAO72KZ0,hugo_smith@gmail.com
4,YIE6IVWPVS7H,theobernard8@gmail.com
5,5JU9BVM2P9E6,sofia.martin52@gmail.com
6,TGXN0GUO4KH2,theojones@gmail.com
7,AKOIXNTM9TMQ,liammartin@gmail.com
8,TX63CFL0UKEY,mateo_silva@gmail.com
9,K9P38QOLDFOR,claraweber4@gmail.com


<a id="6"></a>
# 6. Trier et limiter : `ORDER BY` et `LIMIT`

Sans `ORDER BY`, l'ordre des lignes n'est pas garanti. Si l'ordre compte pour vous,
il faut le demander explicitement.

```sql
SELECT colonne
FROM ma_table
ORDER BY colonne_de_tri [ASC | DESC]
LIMIT n
```

- `ASC` = croissant (valeur par défaut, on peut l'omettre)
- `DESC` = décroissant
- `LIMIT n` = ne renvoyer que les `n` premières lignes après le tri

In [75]:
%%sql
-- Les 10 clients les plus fortunés
SELECT
    customer_id,
    country_code,
    premium_tier,
    total_asset_eur
FROM customers
ORDER BY total_asset_eur DESC
LIMIT 10

,customer_id,country_code,premium_tier,total_asset_eur
0,U19PA8PJ7KW6,GB,gold,"368,335.63"
1,ESUBR99P9YKN,FR,gold,"321,551.67"
2,AFPP7SDWYQTU,DE,gold,"317,428.71"
3,M4IWTAQLDTMX,IT,gold,"307,488.18"
4,2595EY4948JO,GB,gold,"305,269.88"
5,ELW8IVI3RQAS,DE,gold,"303,850.70"
6,5VZTG12CYVQU,FR,gold,"295,412.37"
7,NCC2KEIVLI77,DE,gold,"290,134.34"
8,8V1ZOCFI4O6L,IT,gold,"287,117.11"
9,YLW4QRWG4O3V,DE,gold,"273,931.38"


In [76]:
%%sql
-- Tri sur plusieurs colonnes : d'abord par pays (A→Z),
-- puis, à l'intérieur de chaque pays, par actifs décroissants.
SELECT
    country_code,
    customer_id,
    total_asset_eur
FROM customers
WHERE country_code IN ('FR', 'DE')
ORDER BY country_code ASC,
         total_asset_eur DESC
LIMIT 12

,country_code,customer_id,total_asset_eur
0,DE,AFPP7SDWYQTU,"317,428.71"
1,DE,ELW8IVI3RQAS,"303,850.70"
2,DE,NCC2KEIVLI77,"290,134.34"
3,DE,YLW4QRWG4O3V,"273,931.38"
4,DE,J6317KO7CTBK,"242,238.38"
5,DE,GJPMH3UQ1AMK,"231,572.32"
6,DE,EPM7ZDNG61SP,"224,173.75"
7,DE,PQPYH5H2O5LT,"223,686.97"
8,DE,NTPLI69VINXR,"222,505.14"
9,DE,RUTSBC23I4S4,"215,908.31"


> `LIMIT` est votre ami
> Prenez l'habitude de terminer vos requêtes d'exploration par `LIMIT 10`. Sur une table
> de plusieurs millions de lignes, une requête sans limite peut saturer la mémoire du notebook.

<a id="7"></a>
# 7. Calculer des indicateurs : les fonctions d'agrégation

Jusqu'ici, une ligne en entrée donnait une ligne en sortie. Les fonctions d'agrégation
changent la donne : elles résument un ensemble de lignes en une seule valeur.

| Fonction | Calcule | Type de colonne |
|---|---|---|
| `COUNT(...)` | un nombre de lignes | n'importe lequel |
| `SUM(...)` | la somme | numérique |
| `AVG(...)` | la moyenne | numérique |
| `MIN(...)` | la plus petite valeur | numérique, date, texte |
| `MAX(...)` | la plus grande valeur | numérique, date, texte |

Sans `GROUP BY`, elles s'appliquent à toute la table et renvoient une seule ligne.

In [77]:
%%sql
SELECT
    COUNT(customer_id)   AS total_clients,
    SUM(total_asset_eur) AS actifs_totaux_eur,
    AVG(total_asset_eur) AS actifs_moyens_eur,
    MIN(total_asset_eur) AS actif_min_eur,
    MAX(total_asset_eur) AS actif_max_eur
FROM customers

,total_clients,actifs_totaux_eur,actifs_moyens_eur,actif_min_eur,actif_max_eur
0,5000,"125,319,468.45","25,063.89",0.00,"368,335.63"


### `COUNT(*)` et `COUNT(colonne)` ne comptent pas la même chose

C'est une distinction subtile mais essentielle :

- `COUNT(*)` compte toutes les lignes, sans regarder leur contenu.
- `COUNT(colonne)` compte les lignes où cette colonne n'est pas `NULL`.

Cette deuxième forme est en fait un outil très pratique : elle permet de compter
« combien de clients ont un abonnement » sans écrire de `WHERE`.

In [79]:
%%sql
SELECT
    COUNT(*)            AS toutes_les_lignes,           -- tous les clients
    COUNT(customer_id)  AS avec_identifiant,            -- identique : jamais NULL
    COUNT(premium_tier) AS avec_abonnement_premium,     -- exclut les NULL → clients premium
    COUNT(verified_at)  AS clients_verifies,            -- exclut les NULL → KYC validé
    COUNT(referral_url) AS avec_url_de_provenance       -- exclut le trafic direct
FROM customers

,toutes_les_lignes,avec_identifiant,avec_abonnement_premium,clients_verifies,avec_url_de_provenance
0,5000,5000,1770,3979,4724


> `AVG` ignore aussi les `NULL`. La moyenne est calculée sur les seules lignes
> renseignées, pas sur le total. Si 200 clients sur 1 000 ont une valeur et que leur
> moyenne est 50, `AVG` renvoie 50 et non 10. Quand cette distinction compte,
> vérifiez le dénominateur avec un `COUNT` de la même colonne.

## Exercice 7.1

L'équipe Finance prépare son rapport mensuel. Elle veut quatre indicateurs globaux :

- le nombre total de clients ;
- le total des dépôts en euros ;
- le total des retraits en euros ;
- la moyenne des actifs par client.

Le tout en une seule requête, avec des noms de colonnes lisibles.

<details>
<summary> Indice</summary>

Quatre fonctions d'agrégation dans un même `SELECT`, séparées par des virgules,
chacune renommée avec `AS`.
</details>

In [ ]:
%%sql
-- Votre code ici

In [80]:
%%sql
-- Correction
SELECT
    COUNT(customer_id)        AS total_clients,
    SUM(total_deposit_eur)    AS total_depots_eur,
    SUM(total_withdrawal_eur) AS total_retraits_eur,
    AVG(total_asset_eur)      AS actifs_moyens_eur
FROM customers

,total_clients,total_depots_eur,total_retraits_eur,actifs_moyens_eur
0,5000,"182,111,311.50","39,609,066.72","25,063.89"


## Exercice 7.2

Reprenez la requête précédente, mais uniquement pour les clients français.
Puis relancez-la pour les clients suisses (`CH`).

Gardez le résultat sous les yeux : la section suivante montre comment obtenir les deux
d'un seul coup.

<details>
<summary>Indice</summary>

Un agrégat et un filtre cohabitent très bien : ajoutez simplement une clause `WHERE`.
</details>

In [ ]:
%%sql
-- Votre code ici

In [81]:
%%sql
-- Correction : la France
SELECT
    COUNT(customer_id)        AS total_clients,
    SUM(total_deposit_eur)    AS total_depots_eur,
    SUM(total_withdrawal_eur) AS total_retraits_eur,
    AVG(total_asset_eur)      AS actifs_moyens_eur
FROM customers
WHERE country_code = 'FR'

,total_clients,total_depots_eur,total_retraits_eur,actifs_moyens_eur
0,1549,"57,922,032.65","13,137,883.42","25,464.28"


In [82]:
%%sql
-- Correction : la Suisse
SELECT
    COUNT(customer_id)        AS total_clients,
    SUM(total_deposit_eur)    AS total_depots_eur,
    SUM(total_withdrawal_eur) AS total_retraits_eur,
    AVG(total_asset_eur)      AS actifs_moyens_eur
FROM customers
WHERE country_code = 'CH'

,total_clients,total_depots_eur,total_retraits_eur,actifs_moyens_eur
0,234,"8,439,918.01","1,706,180.34","25,443.82"


<a id="8"></a>
# 8. Calculer par groupe : `GROUP BY`

Dans l'exercice précédent, vous avez lancé la même requête deux fois en changeant le pays.
Avec dix pays, ce serait dix requêtes et un copier-coller pour chaque nouveau marché.

`GROUP BY` résout exactement ce problème. Il dit à SQL :

> range les lignes en paquets selon cette colonne, puis calcule les agrégats paquet par paquet.

```sql
SELECT   colonne_de_groupe,
         AGREGAT(autre_colonne)
FROM     ma_table
GROUP BY colonne_de_groupe
```

La règle d'or : toute colonne du `SELECT` qui n'est pas dans une fonction
d'agrégation doit apparaître dans le `GROUP BY`. Sinon, SQL ne saurait pas quelle
valeur afficher pour le groupe et renvoie une erreur.

In [85]:
%%sql
-- Les deux résultats de l'exercice 7.2 en une seule requête…
SELECT
    country_code,
    COUNT(customer_id)        AS total_clients,
    SUM(total_deposit_eur)    AS total_depots_eur,
    AVG(total_asset_eur)      AS actifs_moyens_eur
FROM customers
WHERE country_code IN ('FR', 'CH')
GROUP BY country_code

,country_code,total_clients,total_depots_eur,actifs_moyens_eur
0,FR,1549,"57,922,032.65","25,464.28"
1,CH,234,"8,439,918.01","25,443.82"


In [86]:
%%sql
-- … et, en retirant le filtre, tous les pays d'un coup.
SELECT
    country_code,
    COUNT(customer_id)   AS total_clients,
    AVG(total_asset_eur) AS actifs_moyens_eur
FROM customers
GROUP BY country_code
ORDER BY total_clients DESC

,country_code,total_clients,actifs_moyens_eur
0,FR,1549,"25,464.28"
1,DE,1110,"25,588.87"
2,GB,799,"25,587.86"
3,ES,432,"23,637.80"
4,IT,389,"24,221.19"
5,CH,234,"25,443.82"
6,BE,201,"25,981.28"
7,NL,155,"22,754.67"
8,PT,89,"17,358.71"
9,LU,42,"27,271.22"


### L'erreur classique : une colonne oubliée dans le `GROUP BY`

La cellule suivante échoue volontairement. Lisez le message d'erreur : il vous dit
précisément quelle colonne pose problème. Apprendre à lire ces messages vous fera gagner
un temps considérable.

In [88]:
%%sql
-- ✗ ERREUR ATTENDUE : employment_status est dans le SELECT
--   mais absente du GROUP BY.
SELECT
    country_code,
    employment_status,
    COUNT(customer_id) AS total_clients
FROM customers
GROUP BY country_code

In [89]:
%%sql
-- ✓ Correction : on groupe sur les deux colonnes.
-- Chaque ligne du résultat = une combinaison (pays, situation professionnelle).
SELECT
    country_code,
    employment_status,
    COUNT(customer_id) AS total_clients
FROM customers
GROUP BY country_code, employment_status
ORDER BY country_code, total_clients DESC
LIMIT 15

,country_code,employment_status,total_clients
0,BE,employee,123
1,BE,business owner,37
2,BE,unemployed,24
3,BE,None,17
4,CH,employee,148
5,CH,business owner,39
6,CH,unemployed,30
7,CH,None,17
8,DE,employee,675
9,DE,business owner,205


## Exercice 8

L'équipe Produit veut comprendre la distribution des abonnements.

Elle demande, par niveau de `premium_tier` (y compris les clients sans abonnement) :
le nombre de clients et la moyenne de leurs actifs.

Triez par nombre de clients décroissant.

*Colonnes attendues :* `premium_tier`, nombre de clients, moyenne des actifs

<details>
<summary> Indice</summary>

« Y compris les clients sans abonnement » signifie : pas de `WHERE`.
Le groupe `NULL` est un groupe comme un autre pour `GROUP BY`.
</details>

In [ ]:
%%sql
-- Votre code ici

In [90]:
%%sql
-- Correction
SELECT
    premium_tier,
    COUNT(customer_id)   AS total_clients,
    AVG(total_asset_eur) AS actifs_moyens_eur
FROM customers
GROUP BY premium_tier
ORDER BY total_clients DESC

,premium_tier,total_clients,actifs_moyens_eur
0,None,3230,"5,278.66"
1,bronze,1041,"30,305.82"
2,silver,445,"86,966.13"
3,gold,284,"133,877.11"


<a id="9"></a>
# 9. Supprimer les doublons : `DISTINCT`

Question simple : dans quels pays Novalto a-t-elle des clients ?

Un `SELECT country_code` renvoie une ligne par client des milliers de lignes, avec
« FR » répété des centaines de fois. `DISTINCT` ne garde que les valeurs uniques.

In [91]:
%%sql
-- ✗ Une ligne par client : illisible
SELECT country_code
FROM customers
LIMIT 10

,country_code
0,IT
1,FR
2,IT
3,NL
4,FR
5,FR
6,GB
7,FR
8,FR
9,DE


In [96]:
%%sql
-- ✓ La liste des pays, sans doublon
SELECT DISTINCT country_code
FROM customers
ORDER BY country_code

,country_code
0,BE
1,CH
2,DE
3,ES
4,FR
5,GB
6,IT
7,LU
8,NL
9,PT


### `DISTINCT` sur plusieurs colonnes

`DISTINCT` porte sur l'ensemble des colonnes sélectionnées, pas sur la première.
Le résultat est la liste des combinaisons uniques.

In [97]:
%%sql
SELECT DISTINCT
    premium_tier,
    employment_status
FROM customers
ORDER BY premium_tier, employment_status

,premium_tier,employment_status
0,bronze,business owner
1,bronze,employee
2,bronze,unemployed
3,bronze,None
4,gold,business owner
5,gold,employee
6,gold,unemployed
7,gold,None
8,silver,business owner
9,silver,employee


### `COUNT(DISTINCT ...)` : compter les valeurs différentes

Combien de pays, et non combien de clients ? On combine les deux fonctions.

In [98]:
%%sql
SELECT
    COUNT(country_code)          AS lignes_avec_un_pays,
    COUNT(DISTINCT country_code) AS nombre_de_pays_differents
FROM customers

,lignes_avec_un_pays,nombre_de_pays_differents
0,5000,10


> `DISTINCT` ou `GROUP BY` ?
> `SELECT DISTINCT country_code FROM customers` et
> `SELECT country_code FROM customers GROUP BY country_code` renvoient exactement la
> même chose. Préférez `DISTINCT` : l'intention (« dédupliquer ») est plus lisible.
> Utilisez `GROUP BY` quand vous calculez aussi un agrégat.

## Exercice 9.1

L'équipe Produit prépare des tests sur mobile. Elle demande la liste des appareils
utilisés lors de la première connexion, sans doublon, triée par ordre alphabétique.

<details>
<summary> Indice</summary>

Une seule colonne, `first_device`, et deux mots-clés.
</details>

In [ ]:
%%sql
-- Votre code ici

In [99]:
%%sql
-- Correction
SELECT DISTINCT first_device
FROM customers
ORDER BY first_device

,first_device
0,Google Pixel 7
1,MacBook Pro
2,Samsung Galaxy S23
3,Windows Desktop
4,Xiaomi Redmi Note 12
5,iPad Air
6,iPhone 14
7,iPhone 15


## Exercice 9.2

L'équipe Conformité doit déclarer dans combien de pays Novalto compte des clients actifs. Un seul chiffre attendu.

<details>
<summary> Indice</summary>

Combinez un filtre sur `account_status` et un `COUNT(DISTINCT ...)`.
</details>

In [ ]:
%%sql
-- Votre code ici

In [100]:
%%sql
-- Correction
SELECT COUNT(DISTINCT country_code) AS nombre_de_pays_actifs
FROM customers
WHERE account_status = 'ACCOUNT_STATUS_ACTIVE'

,nombre_de_pays_actifs
0,10


<a id="10"></a>
# 10. Filtrer sur un agrégat : `HAVING`

L'équipe Marketing veut la liste des pays où Novalto compte plus de 100 clients.

Le réflexe naturel serait d'écrire `WHERE COUNT(customer_id) > 100`. Essayons.

In [101]:
%%sql
-- ✗ ERREUR ATTENDUE : on ne peut pas filtrer sur un agrégat dans WHERE.
SELECT
    country_code,
    COUNT(customer_id) AS total_clients
FROM customers
WHERE COUNT(customer_id) > 100
GROUP BY country_code

Pourquoi cette erreur ? Une question de chronologie.

`WHERE` s'exécute avant le regroupement, ligne par ligne. À ce moment-là, les groupes
n'existent pas encore et le `COUNT` n'a pas encore été calculé : SQL ne peut donc pas
filtrer dessus.

`HAVING` s'exécute après le `GROUP BY`, une fois les agrégats calculés.

> La règle en une phrase :
> `WHERE` filtre les lignes, `HAVING` filtre les groupes.

In [102]:
%%sql
-- ✓ HAVING filtre les groupes, une fois le COUNT calculé
SELECT
    country_code,
    COUNT(customer_id) AS total_clients
FROM customers
GROUP BY country_code
HAVING COUNT(customer_id) > 100
ORDER BY total_clients DESC

,country_code,total_clients
0,FR,1549
1,DE,1110
2,GB,799
3,ES,432
4,IT,389
5,CH,234
6,BE,201
7,NL,155


### Les deux ensemble

`WHERE` et `HAVING` ne s'opposent pas : ils travaillent à deux étages différents et se
combinent très bien dans une même requête.

In [103]:
%%sql
-- Les pays comptant plus de 50 clients ACTIFS
SELECT
    country_code,
    COUNT(customer_id) AS clients_actifs
FROM customers
WHERE account_status = 'ACCOUNT_STATUS_ACTIVE'   -- 1. garde les lignes des clients actifs
GROUP BY country_code                            -- 2. regroupe par pays
HAVING COUNT(customer_id) > 50                   -- 3. ne garde que les gros pays
ORDER BY clients_actifs DESC                     -- 4. trie

,country_code,clients_actifs
0,FR,1084
1,DE,775
2,GB,580
3,ES,303
4,IT,262
5,CH,163
6,BE,131
7,NL,113
8,PT,56


## Exercice 10

L'équipe Produit veut identifier les niveaux d'abonnement qui concentrent les clients
les plus fortunés.

Pour les clients avec un abonnement premium uniquement, elle demande les niveaux de
`premium_tier` dont la moyenne d'actifs dépasse 70 000 €. Pour chaque niveau retenu :
le nombre de clients et la moyenne des actifs, triés par moyenne décroissante.

<details>
<summary> Indice</summary>

Trois étapes s'enchaînent : un `WHERE` pour exclure les non-premium, un `GROUP BY`
sur le niveau, un `HAVING` sur la moyenne.
</details>

In [ ]:
%%sql
-- Votre code ici

In [104]:
%%sql
-- Correction
SELECT
    premium_tier,
    COUNT(customer_id)   AS total_clients,
    AVG(total_asset_eur) AS actifs_moyens_eur
FROM customers
WHERE premium_tier IS NOT NULL
GROUP BY premium_tier
HAVING AVG(total_asset_eur) > 70000
ORDER BY actifs_moyens_eur DESC

,premium_tier,total_clients,actifs_moyens_eur
0,gold,284,"133,877.11"
1,silver,445,"86,966.13"


> Alias dans `HAVING` : attention à la portabilité.
> DuckDB accepte `HAVING actifs_moyens_eur > 70000` (l'alias défini dans le `SELECT`).
> C'est confortable, mais PostgreSQL le refuse, car `HAVING` est logiquement évalué
> avant `SELECT`. Pour écrire du SQL qui fonctionne partout, répétez l'agrégat :
> `HAVING AVG(total_asset_eur) > 70000`.

<a id="11"></a>
# 11. L'ordre d'exécution : la clé qui explique tout

Vous avez rencontré plusieurs comportements qui semblaient arbitraires :
`WHERE` refuse les agrégats, `HAVING` les accepte, l'alias marche ici mais pas là.

Tout s'explique par une seule chose : SQL ne s'exécute pas dans l'ordre où on l'écrit.

| Ordre d'**écriture** | Ordre d'**exécution** | Ce qui se passe |
|---|---|---|
| 1. `SELECT` | 5️⃣ | choisit les colonnes, applique les alias `AS` |
| 2. `FROM` | 1️⃣ | va chercher la table |
| 3. `WHERE` | 2️⃣ | filtre les lignes |
| 4. `GROUP BY` | 3️⃣ | forme les groupes |
| 5. `HAVING` | 4️⃣ | filtre les groupes |
| 6. `ORDER BY` | 6️⃣ | trie le résultat |
| 7. `LIMIT` | 7️⃣ | tronque le résultat |

Ce que cette table explique :

- `WHERE` (2️⃣) ne peut pas voir un agrégat calculé en 3️⃣–4️⃣. → il faut `HAVING`.
- `HAVING` (4️⃣) s'exécute avant `SELECT` (5️⃣), donc avant que l'alias n'existe.
  → en SQL standard, on y répète l'agrégat.
- `ORDER BY` (6️⃣) s'exécute après `SELECT` (5️⃣). → il peut, lui, utiliser les alias.
- `LIMIT` (7️⃣) est vraiment le dernier. → il tronque après le tri, jamais avant.

Gardez cette table sous la main : elle répond à la majorité des « pourquoi ça ne marche pas ? »
des premiers mois.

In [105]:
%%sql
-- Illustration : l'alias `actifs_moyens_eur` fonctionne dans ORDER BY (6️⃣, après SELECT)
-- mais pas dans WHERE (2️⃣, bien avant).
SELECT
    country_code,
    AVG(total_asset_eur) AS actifs_moyens_eur
FROM customers
GROUP BY country_code
ORDER BY actifs_moyens_eur DESC
LIMIT 5

,country_code,actifs_moyens_eur
0,LU,"27,271.22"
1,BE,"25,981.28"
2,DE,"25,588.87"
3,GB,"25,587.86"
4,FR,"25,464.28"


<a id="12"></a>
# 12. Exercices de synthèse

Ces trois exercices mobilisent l'ensemble du cours. Prenez le temps de décomposer chaque
énoncé avant d'écrire : *quelles lignes ? quels groupes ? quels calculs ? quel tri ?*

## Synthèse 1 Rapport d'acquisition

La direction veut un tableau de bord par pays, limité aux clients vérifiés
(ceux qui ont passé le KYC).

Pour chaque pays, affichez : le nombre de clients, le nombre de clients premium,
la moyenne des actifs. Ne gardez que les pays comptant au moins 100 clients vérifiés,
et triez par nombre de clients décroissant.

<details>
<summary> Indice</summary>

Le nombre de clients premium se calcule sans `WHERE` supplémentaire :
souvenez-vous que `COUNT(premium_tier)` ignore les `NULL`.
</details>

In [ ]:
%%sql
-- Votre code ici

In [106]:
%%sql
-- Correction
SELECT
    country_code,
    COUNT(customer_id)   AS clients_verifies,
    COUNT(premium_tier)  AS clients_premium,
    AVG(total_asset_eur) AS actifs_moyens_eur
FROM customers
WHERE verified_at IS NOT NULL
GROUP BY country_code
HAVING COUNT(customer_id) >= 100
ORDER BY clients_verifies DESC

,country_code,clients_verifies,clients_premium,actifs_moyens_eur
0,FR,1223,460,"32,251.98"
1,DE,873,311,"32,535.67"
2,GB,660,222,"30,976.82"
3,ES,340,120,"30,033.91"
4,IT,299,110,"31,511.85"
5,CH,195,68,"30,532.58"
6,BE,162,55,"32,236.03"
7,NL,128,47,"27,554.48"


## Synthèse 2 Performance des canaux d'acquisition

L'équipe Growth veut comparer deux canaux : Reddit et Google.

Pour chacun, elle veut le nombre de clients et la moyenne des actifs. Une seule requête.

<details>
<summary> Indice</summary>

Une piste : filtrez avec `LIKE` sur les deux canaux, puis groupez sur une expression
qui extrait la source. `CASE WHEN referral_url LIKE '%reddit%' THEN 'reddit' ELSE 'google' END`
crée une colonne à la volée — c'est un avant-goût de la suite du parcours.
</details>

In [ ]:
%%sql
-- Votre code ici

In [107]:
%%sql
-- Correction
SELECT
    CASE
        WHEN referral_url LIKE '%reddit%' THEN 'reddit'
        WHEN referral_url LIKE '%google%' THEN 'google'
    END                  AS canal,
    COUNT(customer_id)   AS total_clients,
    AVG(total_asset_eur) AS actifs_moyens_eur
FROM customers
WHERE referral_url LIKE '%reddit%'
   OR referral_url LIKE '%google%'
GROUP BY canal
ORDER BY total_clients DESC

,canal,total_clients,actifs_moyens_eur
0,google,1725,"24,986.18"
1,reddit,1110,"25,528.05"


## Synthèse 3 Comptes à relancer

Le Support veut relancer les comptes bloqués : ceux qui sont en onboarding ou en
recovery, et qui n'ont pas encore validé leur identité.

Affichez le nombre de comptes concernés par pays et par statut, triés par pays puis
par nombre décroissant.

<details>
<summary> Indice</summary>

Deux conditions à combiner : un `OR` entre les statuts ( parenthèses !) et un `AND`
sur le KYC. Puis un `GROUP BY` sur deux colonnes.
</details>

In [ ]:
%%sql
-- Votre code ici

In [108]:
%%sql
-- Correction
SELECT
    country_code,
    account_status,
    COUNT(customer_id) AS comptes_a_relancer
FROM customers
WHERE account_status IN ('ACCOUNT_STATUS_ONBOARDING', 'ACCOUNT_STATUS_RECOVERY')
  AND verified_at IS NULL
GROUP BY country_code, account_status
ORDER BY country_code, comptes_a_relancer DESC

,country_code,account_status,comptes_a_relancer
0,BE,ACCOUNT_STATUS_ONBOARDING,39
1,CH,ACCOUNT_STATUS_ONBOARDING,39
2,DE,ACCOUNT_STATUS_ONBOARDING,237
3,ES,ACCOUNT_STATUS_ONBOARDING,92
4,FR,ACCOUNT_STATUS_ONBOARDING,326
5,GB,ACCOUNT_STATUS_ONBOARDING,139
6,IT,ACCOUNT_STATUS_ONBOARDING,90
7,LU,ACCOUNT_STATUS_ONBOARDING,8
8,NL,ACCOUNT_STATUS_ONBOARDING,27
9,PT,ACCOUNT_STATUS_ONBOARDING,24


<a id="cheatsheet"></a>
# Félicitations !

Vous savez maintenant lire, filtrer, agréger et regrouper des données avec SQL.
C'est le socle : la grande majorité des requêtes du quotidien n'utilise rien de plus.

---

## Aide-mémoire

### La requête complète, dans l'ordre d'écriture

```sql
SELECT   colonne, AGREGAT(colonne) AS alias   -- quelles colonnes ?      (exécuté 5e)
FROM     table                                -- quelle table ?          (exécuté 1er)
WHERE    condition_sur_les_lignes             -- quelles lignes ?        (exécuté 2e)
GROUP BY colonne                              -- quels groupes ?         (exécuté 3e)
HAVING   condition_sur_les_groupes            -- quels groupes garder ?  (exécuté 4e)
ORDER BY colonne DESC                         -- quel tri ?              (exécuté 6e)
LIMIT    10                                   -- combien de lignes ?     (exécuté 7e)
```

### Filtrer

```sql
WHERE country_code = 'FR'                      -- égalité (texte entre '')
WHERE total_asset_eur >= 10000                 -- comparaison numérique
WHERE country_code IN ('FR', 'DE', 'GB')       -- fait partie d'une liste
WHERE total_asset_eur BETWEEN 1000 AND 5000    -- intervalle, bornes incluses
WHERE verified_at IS NULL                      -- valeur absente  (jamais = NULL)
WHERE premium_tier IS NOT NULL                 -- valeur présente (jamais != NULL)
WHERE email LIKE '%gmail.com'                  -- finit par
WHERE referral_url LIKE '%reddit%'             -- contient
WHERE (a = 1 OR a = 2) AND b = 3               -- parenthèses dès qu'on mélange AND/OR
```

### Agréger

```sql
COUNT(*)              -- toutes les lignes
COUNT(colonne)        -- lignes où la colonne n'est pas NULL
COUNT(DISTINCT col)   -- nombre de valeurs différentes
SUM(col)  AVG(col)  MIN(col)  MAX(col)
```

### Les cinq pièges à retenir

| Piège | Symptôme | Solution |
|---|---|---|
| `= NULL` | 0 ligne, aucune erreur | `IS NULL` / `IS NOT NULL` |
| `AND` prioritaire sur `OR` | résultat faux, aucune erreur | parenthèses, ou `IN (…)` |
| Virgule oubliée dans `SELECT` | une colonne disparaît | relire la liste des colonnes |
| Colonne absente du `GROUP BY` | erreur explicite | l'ajouter au `GROUP BY` |
| `WHERE` sur un agrégat | erreur explicite | utiliser `HAVING` |

---

## Pour aller plus loin

Les prochaines briques, dans l'ordre où elles servent le plus souvent :

1. `JOIN` croiser plusieurs tables. La compétence la plus rentable après ce cours.
2. `CASE WHEN` créer des colonnes conditionnelles (vous en avez vu un aperçu en synthèse 2).
3. Fonctions de date `DATE_DIFF`, `DATE_TRUNC` : calculer un âge, agréger par mois.
4. CTE (`WITH ... AS`) découper une requête complexe en étapes lisibles.
5. Fonctions fenêtre `ROW_NUMBER`, `RANK`, `SUM() OVER (...)` : classements et cumuls.

Ressources
- [Documentation DuckDB](https://duckdb.org/docs/sql/introduction) le moteur utilisé ici
- [SQL Tutorial Mode Analytics](https://mode.com/sql-tutorial/) exercices interactifs
- [Select Star SQL](https://selectstarsql.com/) un livre en ligne, gratuit et bien fait

---

*Ce notebook est distribué sous licence MIT. Les données sont synthétiques et fictives.*